# Vector Database

Make sure you have a .env file with the following fields: 

- PGHOST=localhost
- PGPORT=5432
- PGDB=rag_db
- PGUSER=postgres
- PGPASSWORD=your_postgres_password

## Import libraries and define paths

In [10]:
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Iterator, Dict, Any, List
import numpy as np

# To import the passwords
import os
#pip install python-dotenv
from dotenv import load_dotenv

# To connect with posgreSQL db
import psycopg2
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values

In [11]:
DOCS_ROOT = Path("docs")
TOPIC_FOLDERS = {"general", "mama"}  # extend later: {"general","mama","prostata",...}
PAGES_JSONL = Path("docs/pages.jsonl")
CHUNKS_JSONL = Path("docs/chunks.jsonl")
#Document metadata structure
@dataclass
class DocMeta:
    doc_id: str #stable unique ID
    topic: str #mama, general, prostata, etc.
    lang: str #language
    source: str #novartis, gepac, etc.
    slug: str #descriptive text
    version: str #version (v1, v2) or year of publication
    path: str #where the file is on disk
    file_hash: str #unique hash to detect changes

In [12]:
load_dotenv(".env")  # path to your secrets file
dbname=os.environ["PGDB"]
user=os.environ["PGUSER"]
password=os.environ["PGPASSWORD"]
host=os.environ['PGHOST']
port=os.environ['PGHOST']


## Load chunk text and chunk vectors

In [ ]:
emb = np.load("docs/chunks_vectors.npy")
emb.shape #(909, 384)

In [ ]:
loaded_chunks = []
with CHUNKS_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        loaded_chunks.append(json.loads(line))

print("Loaded chunks:", len(loaded_chunks)) #909

## Add embeddings to PostgreSQL vector db (bulk insert)

In [ ]:
# 1) Connect to YOUR database (not "postgres" unless that's where your table is)
conn = psycopg2.connect(
    dbname=dbname,          # <-- change to your DB name
    user=user,
    password=password, #Don't upload to GitHub
    host=host,
    port=port
)

# 2) Make psycopg2 understand the pgvector type
register_vector(conn)

cur = conn.cursor()

# 3) Suppose you already have:
# texts = [c["text"] for c in all_chunks]
# emb = emb_model.encode(...)

# emb is usually a numpy array shape (N, 384).
# Convert each row to a plain Python list (pgvector adapter handles it well)´
texts = loaded_chunks.copy()
#rows = [(t, e.tolist()) for t, e in zip(texts, emb)]
# loaded_chunks is a list of dicts, emb is (N, 384)
rows = [(c["text"], emb[i].tolist()) for i, c in enumerate(loaded_chunks)]
print(rows[1])

# 4) Bulk insert (fast)
'''Assuming that you table structure is: 
    CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT,
    embedding VECTOR(1536)
    );
'''
execute_values(
    cur,
    "INSERT INTO documents (content, embedding) VALUES %s",
    rows
)

conn.commit()
cur.close()
conn.close()

print(f"Inserted {len(rows)} rows.")

### Sanity check

In [9]:
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host)
register_vector(conn)
cur = conn.cursor()

cur.execute("SELECT id, left(content, 400), embedding FROM documents LIMIT 2;")
print(cur.fetchall())

cur.close()
conn.close()

[(1, 'Vol.:(0123456789) 1 3 Clinical and Translational Oncology (2023) 25:2665–2678 https://doi.org/10.1007/s12094-023-03203-8 CLINICAL GUIDES IN\xa0ONCOLOGY SEOM–GEICAM–SOLTI clinical guidelines in\xa0advanced breast cancer (2022) Jose\xa0Angel\xa0Garcia‑Saenz1\u200a \xa0· Isabel\xa0Blancas2\u200a \xa0· Isabel\xa0Echavarria3\u200a \xa0· Carmen\xa0Hinojo4\u200a \xa0· Mireia\xa0Margeli5\xa0· Fernando\xa0Moreno1\u200a \xa0· Sonia\xa0Pernas6\u200a \xa0· Teresa\xa0Ramon\xa0y\xa0Cajal7\u200a \xa0· Nuria\xa0', array([-5.79986349e-02, -8.59923959e-02,  1.97257027e-02,  3.19715515e-02,
        9.42024309e-03, -5.08584431e-04, -3.04730274e-02,  5.03634773e-02,
        2.17071306e-02, -4.89296839e-02,  9.60904581e-04, -3.74150351e-02,
       -6.54999912e-02,  1.22926245e-02,  8.46127421e-03, -1.27214612e-02,
       -2.88276747e-02, -3.69683392e-02, -6.94779083e-02,  4.71994542e-02,
       -5.87821864e-02,  9.24992934e-03,  2.94780955e-02,  3.47960629e-02,
       -4.33659367e-02, -1.47012085e-01,

In [7]:
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host)
register_vector(conn)
cur = conn.cursor()

cur.execute("SELECT id, content, embedding FROM documents WHERE id = 6;")
print(cur.fetchall())

cur.close()
conn.close()

[(6, '2667 Clinical and Translational Oncology (2023) 25:2665–2678 1 3 Microsatellite instability (MSI-high), an infrequent biomarker in breast cancer (1.7%) [I, C], could be considered if therapy with pembrolizumab is available.\n\nOther infrequent biomarkers in breast cancer (<\u20090.1%) that might also be tested, are the NTRK fusions/translocations, since their detection are associated to a high efficacy of NKTR inhibitors irrespectively of the type of primary tumor (agnostic indication) [I, C].\n\nAccording to the ESCAT guidelines [4] that include high number of level II alterations, tumor multigene next-generation sequencing (NGS) and circulating tumor DNA (ctDNA) genomic profiling tests are still not routinely recommended, although they should be offered to MBC patients if it may change treatment or enable their inclusion in clinical trials [V, B].\n\nFigure\xa01 briefly summarizes the different variables to be considered for decision-making on the treatment of ABC patients.\n\n

# Semantic search

Make sure: 
```-- embedding dim must match your model (384)
ALTER TABLE documents
  ALTER COLUMN embedding TYPE vector(384);

-- fast ANN index
CREATE INDEX IF NOT EXISTS documents_embedding_hnsw
ON documents
USING hnsw (embedding vector_cosine_ops);```


In [4]:
# If needed:
#Install + load model

# !pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer #Uses pytorch, not tensorflow

EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
#Embedding size = 384
#Maximum input length = 512 tokens
emb_model = SentenceTransformer(EMB_MODEL_NAME)

c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mamen\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pytho

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector

def semantic_search(query: str, k: int = 5):
    # 1) Embed the query (same model, same normalization)
    q_emb = emb_model.encode([query], normalize_embeddings=True)[0].tolist()

    # 2) Connect
    conn = psycopg2.connect(
    dbname=dbname,          
    user=user,
    password=password,
    host=host,
)
    register_vector(conn)
    cur = conn.cursor()

    # 3) Query Postgres
    # <=> = cosine distance when using vector_cosine_ops
    # "Give me the k rows whose embeddings are most similar to this query embedding.
    # <=>   = cosine distance
    # %s::vector means treat this vector as a pgvector
    # Sort rows by similarity to the query embedding.
    cur.execute(
        """
        SELECT id,
               content,
               (embedding <=> %s::vector) AS distance
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
        """,
        (q_emb, q_emb, k)
    )
    rows = cur.fetchall()

    cur.close()
    conn.close()

    # 4) Format results (distance smaller = better)
    results = []
    for rank, (doc_id, content, distance) in enumerate(rows, start=1):
        results.append({
            "rank": rank,
            "id": doc_id,
            "distance": float(distance),
            "content": content
        })
    return results


In [14]:
query = "efectos secundarios de la quimioterapia"
top_k = semantic_search(query, k=5)

for r in top_k:
    print(f"\n#{r['rank']} id={r['id']} distance={r['distance']:.4f}")
    print(r["content"][:300], "...")



#1 id=519 distance=0.2657
Todo ello también puede afectar a la imagen corporal y al funcionamiento sexual, por lo que la radioterapia puede alterar la ca­lidad de vida relacionada con la salud incluido el bienestar sexual.

Impacto de la quimioterapia Los efectos adversos de la quimioterapia son importantes, pero se ha de te ...

#2 id=582 distance=0.2674
www.gepac.es 3 INTRODUCCIÓN La quimioterapia forma parte del tratamiento de la mayoría de las enfermedades oncológicas en algún momento de su evolución.

Los fármacos quimioterápicos pueden administrarse con distintos objetivos: eliminar la enfermedad micrometastásica para evitar recidivas futuras ( ...

#3 id=609 distance=0.2757
Toxicidad por quimioterapia 20 10.

TOXICIDAD UNGUEAL Es frecuente la aparición de onicodistrofi a (alteración del color y del crecimiento de las uñas) y de onicolisis (destrucción de la uña).

Pueden aparecer también áreas de pigmentación en líneas o bandas.

Existen diversos fármacos quimioterápic ...

#4 